In [1]:
import os
import pandas as pd

def find_project_root():
    current = os.getcwd()
    while True:
        if os.path.basename(current) == "broken_dataset_repair":
            return current
        parent = os.path.dirname(current)
        if parent == current:
            raise RuntimeError("❌ Could not locate project root")
        current = parent

BASE_DIR = find_project_root()

print("✅ BASE_DIR:", BASE_DIR)

INPUT_PATH = os.path.join(BASE_DIR, "reports/issue_reports/detected_issues.csv")
PROFILE_PATH = os.path.join(BASE_DIR, "reports/profiling_reports/dataset_profile.csv")
OUTPUT_PATH = os.path.join(BASE_DIR, "reports/repair_plan/repair_plan.csv")

✅ BASE_DIR: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair


In [2]:
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"Missing detected_issues: {INPUT_PATH}")

if not os.path.exists(PROFILE_PATH):
    raise FileNotFoundError(f"Missing dataset_profile: {PROFILE_PATH}")

detected_issues = pd.read_csv(INPUT_PATH)
dataset_profile = pd.read_csv(PROFILE_PATH)

# 🔥 CRITICAL NORMALIZATION (THIS FIXES YOUR BUG)
detected_issues.columns = detected_issues.columns.str.strip().str.lower()
dataset_profile.columns = dataset_profile.columns.str.strip().str.lower()

detected_issues["column_name"] = detected_issues["column_name"].str.strip().str.lower()
dataset_profile["column_name"] = dataset_profile["column_name"].str.strip().str.lower()

if detected_issues.empty:
    raise ValueError("detected_issues is empty")

if dataset_profile.empty:
    raise ValueError("dataset_profile is empty")

print("✅ Inputs loaded")

✅ Inputs loaded


In [3]:
print("\n🔍 DEBUG MATCH CHECK")

print("\nDetected issues columns:")
print(sorted(detected_issues["column_name"].unique()))

print("\nProfile columns:")
print(sorted(dataset_profile["column_name"].unique()))


🔍 DEBUG MATCH CHECK

Detected issues columns:
['native-country', 'occupation', 'workclass']

Profile columns:
['age', 'capital-gain', 'capital-loss', 'education', 'educational-num', 'fnlwgt', 'gender', 'hours-per-week', 'id', 'income', 'marital-status', 'native-country', 'occupation', 'race', 'relationship', 'workclass']


In [4]:
records = []

for _, row in detected_issues.iterrows():

    col = str(row["column_name"]).strip().lower()
    issue = row["issue_type"]

    profile_row = dataset_profile[
        dataset_profile["column_name"] == col
    ]

    # skip if not found (safe)
    if profile_row.empty:
        continue

    dtype = str(profile_row["dtype"].values[0]).lower()

    # =========================
    # MISSING VALUES
    # =========================
    if issue == "missing_values":

        action = "impute_missing"

        if "int" in dtype or "float" in dtype:
            strategy = "median"
        else:
            strategy = "mode"

        priority = "high"
        confidence = 0.9
        details = "Missing values detected"

    # =========================
    # HIGH CARDINALITY
    # =========================
    elif issue == "high_cardinality":

        action = "evaluate_high_cardinality"
        strategy = "no_action"
        priority = "medium"
        confidence = 0.7
        details = "High cardinality column"

    # =========================
    # IDENTIFIER
    # =========================
    elif issue == "id_column":

        action = "mark_as_identifier"
        strategy = "no_action"
        priority = "low"
        confidence = 0.5
        details = "Identifier column"

    else:
        continue

    records.append({
        "column_name": col,
        "issue_type": issue,
        "recommended_action": action,
        "strategy": strategy,
        "action_priority": priority,
        "confidence": confidence,
        "details": details
    })

repair_plan = pd.DataFrame(records)

In [5]:
if repair_plan.empty:
    raise ValueError("repair_plan is empty")

REQUIRED_OUTPUT_COLUMNS = [
    "column_name",
    "issue_type",
    "recommended_action",
    "strategy",
    "action_priority",
    "confidence",
    "details"
]

if list(repair_plan.columns) != REQUIRED_OUTPUT_COLUMNS:
    raise ValueError("repair_plan schema mismatch")

if repair_plan.isnull().any().any():
    raise ValueError("repair_plan contains null values")

repair_plan = repair_plan.sort_values(by=["column_name"]).reset_index(drop=True)

print("✅ repair_plan validated")

✅ repair_plan validated


In [6]:
print("\n===== REPAIR PLAN AUDIT =====")

print(f"Total actions: {len(repair_plan)}")

print("\nActions per issue_type:")
print(repair_plan["issue_type"].value_counts())

print("\nStrategies used:")
print(repair_plan["strategy"].value_counts())


===== REPAIR PLAN AUDIT =====
Total actions: 3

Actions per issue_type:
issue_type
missing_values    3
Name: count, dtype: int64

Strategies used:
strategy
mode    3
Name: count, dtype: int64


In [7]:
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

repair_plan.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ repair_plan saved to: {OUTPUT_PATH}")


✅ repair_plan saved to: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/reports/repair_plan/repair_plan.csv
